# 00 -- Vertical Slice: End-to-End Hello World

**Objective:** Validate that all system components connect correctly before optimising any individual part.

This notebook walks through a minimal end-to-end pipeline:
1. Download a small set of PubMed abstracts
2. Generate embeddings with a biomedical embedding model
3. Index them in Qdrant
4. Submit a clinical query, retrieve relevant documents, and generate a response with an LLM

**Correctness matters more than quality at this stage.** The goal is a working pipeline from end to end.

---

### Prerequisites
- `docker compose up -d` (Qdrant running at localhost:6333)
- `uv sync --extra dev` (dependencies installed)
- `.env` configured (at minimum, NCBI_EMAIL)

In [1]:
# Quick check that services are running
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Check Qdrant
from qdrant_client import QdrantClient
qdrant = QdrantClient(host="localhost", port=6333)
print(f"Qdrant connected: {qdrant.get_collections()}")

# Check PubMed email
assert os.getenv("NCBI_EMAIL"), "NCBI_EMAIL is not set in .env"
print("NCBI_EMAIL configured")

Qdrant connected: collections=[CollectionDescription(name='pubmed_vertical_slice')]
NCBI_EMAIL configured


## Step 1: Download PubMed abstracts

We use BioPython (Entrez) to search and download a small set of abstracts.
In production this will be a full ingestion pipeline, but here we only need 10-20 papers.

In [2]:
from Bio import Entrez, Medline
from typing import List, Dict

Entrez.email = os.getenv("NCBI_EMAIL")
api_key = os.getenv("NCBI_API_KEY")
if api_key:
    Entrez.api_key = api_key


def fetch_pubmed_abstracts(
    query: str, max_results: int = 15
) -> List[Dict[str, str]]:
    """Search PubMed and return abstracts with metadata."""
    # Fetch IDs
    handle = Entrez.esearch(db="pubmed", term=query, retmax=max_results, sort="relevance")
    record = Entrez.read(handle)
    handle.close()
    ids = record["IdList"]
    print(f"Found {len(ids)} papers for query: '{query}'")

    if not ids:
        return []

    # Download details
    handle = Entrez.efetch(db="pubmed", id=ids, rettype="medline", retmode="text")
    records = list(Medline.parse(handle))
    handle.close()

    papers = []
    for r in records:
        abstract = r.get("AB", "")
        if abstract:  # Only include papers with an abstract
            papers.append({
                "pmid": r.get("PMID", ""),
                "title": r.get("TI", ""),
                "abstract": abstract,
                "authors": "; ".join(r.get("AU", [])),
                "journal": r.get("JT", ""),
                "year": r.get("DP", "")[:4],
            })

    print(f"Downloaded {len(papers)} papers with abstracts")
    return papers


# Download a sample set of papers
papers = fetch_pubmed_abstracts("type 2 diabetes treatment guidelines 2024", max_results=15)

# Quick preview
for p in papers[:3]:
    print(f"\n[{p['pmid']}] {p['title'][:80]}...")
    print(f"   {p['abstract'][:150]}...")

Found 15 papers for query: 'type 2 diabetes treatment guidelines 2024'
Downloaded 15 papers with abstracts

[38472622] Early onset type 2 diabetes mellitus: an update....
   The incidence and prevalence of type 2 diabetes mellitus (T2DM) in young individuals (aged <40 years) have significantly increased in recent years, ap...

[39675348] ISPAD Clinical Practice Consensus Guidelines 2024: Type 2 Diabetes in Children a...
   Youth-onset type 2 diabetes (T2D) results from genetic, environmental, and metabolic causes that differ among individuals and populations. This chapte...

[39084811] Diagnosis and Treatment of Hyperglycemia in Pregnancy: Type 2 Diabetes Mellitus ...
   Hyperglycemia in pregnancy due to pre-existing Type 2 diabetes mellitus (T2DM) and gestational diabetes mellitus (GDM) is rising globally with increas...


## Step 2: Generate embeddings

We use a biomedical embedding model to convert abstracts into dense vectors.
We start with `PubMedBERT`; a comparative benchmark across alternatives will be run in Phase 1.

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the embedding model
# Alternatives to evaluate in Phase 1: "dmis-lab/biobert-base-cased-v1.2", "BAAI/bge-base-en-v1.5"
EMBEDDING_MODEL = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
print(f"Loading embedding model: {EMBEDDING_MODEL}")

embed_model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Model loaded | Dimension: {embed_model.get_sentence_embedding_dimension()}")

# Generate embeddings for the abstracts
texts = [p["abstract"] for p in papers]
embeddings = embed_model.encode(texts, show_progress_bar=True, batch_size=8)

print(f"\nGenerated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}")

Skipping import of cpp extensions due to incompatible torch version 2.11.0+cu130 for torchao version 0.16.0             Please see https://github.com/pytorch/ao/issues/2919 for more info
No sentence-transformers model found with name microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract. Creating a new one with mean pooling.


Loading embedding model: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded | Dimension: 768


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Generated 15 embeddings of dimension 768


## Step 3: Index in Qdrant

We create a collection in Qdrant and upload the embeddings together with their metadata.
This is the RAG knowledge base.

In [4]:
from qdrant_client.models import Distance, VectorParams, PointStruct

COLLECTION = "pubmed_vertical_slice"  # Temporary name for this test run
VECTOR_DIM = embeddings.shape[1]

# Recreate the collection (clears it if it already exists)
if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=VECTOR_DIM, distance=Distance.COSINE),
)
print(f"Collection '{COLLECTION}' created (dim={VECTOR_DIM})")

# Upload points with metadata
points = [
    PointStruct(
        id=i,
        vector=embeddings[i].tolist(),
        payload={
            "pmid": papers[i]["pmid"],
            "title": papers[i]["title"],
            "abstract": papers[i]["abstract"],
            "authors": papers[i]["authors"],
            "journal": papers[i]["journal"],
            "year": papers[i]["year"],
        },
    )
    for i in range(len(papers))
]

qdrant.upsert(collection_name=COLLECTION, points=points)

info = qdrant.get_collection(COLLECTION)
print(f"{info.points_count} documents indexed in Qdrant")

Collection 'pubmed_vertical_slice' created (dim=768)
15 documents indexed in Qdrant


## Step 4: Semantic search (Retrieval)

We submit a clinical query, convert it to an embedding, and retrieve the most relevant documents.

In [15]:
# Test query
query = "What are the risks of using GLP-1 receptor agonists before undergoing upper gastrointestinal endoscopy?"

# Embed the query
query_embedding = embed_model.encode(query).tolist()

# Search Qdrant
TOP_K = 3
results = qdrant.query_points(
    collection_name=COLLECTION,
    query=query_embedding,
    limit=TOP_K,
)

print(f"Query: {query}\n")
print(f"Top {TOP_K} retrieved documents:\n")
for i, hit in enumerate(results.points):
    print(f"  [{i+1}] Score: {hit.score:.4f} | PMID: {hit.payload['pmid']}")
    print(f"      {hit.payload['title'][:100]}...")
    print(f"      {hit.payload['abstract']}...")
    print()

Query: What are the risks of using GLP-1 receptor agonists before undergoing upper gastrointestinal endoscopy?

Top 3 retrieved documents:

  [1] Score: 0.9482 | PMID: 39096914
      Gastrointestinal effects of GLP-1 receptor agonists: mechanisms, management, and future directions....
      The availability of glucagon-like peptide-1 (GLP-1) receptor agonists (RAs) such as liraglutide and semaglutide, and a GLP-1 and glucose dependent insulinotropic polypeptide coagonist (tirzepatide) represents a paradigm shift in the management of both type 2 diabetes and obesity. There is now considerable attention, including in the public media, on the effect of both long-acting and short-acting GLP-1RAs to delay gastric emptying. Although slowed gastric emptying is integral to reducing post-prandial blood glucose responses in type 2 diabetes, marked slowing of gastric emptying might also increase the propensity for longer intragastric retention of food, with a consequent increased risk of aspirati

## Step 5: Augmented generation (RAG)

We construct a prompt that includes the retrieved documents as context
and ask the LLM to generate a grounded response.

**For this hello world we use a small model** (or a free API).
Phases 2-3 will use the fine-tuned model.

> Note: If you do not have a GPU, you can use a quantised model (GGUF) via llama-cpp-python,
> or the free Hugging Face Inference API.

In [22]:
def build_augmented_prompt(query: str, retrieved_docs: list) -> str:
    """Construct an augmented prompt with retrieved document context."""
    context_parts = []
    for i, hit in enumerate(retrieved_docs):
        context_parts.append(
            f"[{i+1}] PMID: {hit.payload['pmid']} | {hit.payload['title']}\n"
            f"{hit.payload['abstract']}\n"
        )
    context = "\n---\n".join(context_parts)

    prompt = f"""You are a clinical assistant grounded in scientific evidence.
Answer the user's query based ONLY on the documents provided below.
Cite sources using [PMID: XXXXX]. If the context is insufficient, say so explicitly.

=== REFERENCE DOCUMENTS ===
{context}
=== END OF DOCUMENTS ===

USER QUERY: {query}

EVIDENCE-BASED RESPONSE:"""

    return prompt


# Build the prompt
augmented_prompt = build_augmented_prompt(query, results.points)

print("Prompt constructed:")
print(f"   Length: {len(augmented_prompt)} characters")
print(f"   Documents in context: {len(results.points)}")
print(f"\n{'=' * 60}")
print(augmented_prompt)
print("...")

Prompt constructed:
   Length: 4181 characters
   Documents in context: 3

You are a clinical assistant grounded in scientific evidence.
Answer the user's query based ONLY on the documents provided below.
Cite sources using [PMID: XXXXX]. If the context is insufficient, say so explicitly.

=== REFERENCE DOCUMENTS ===
[1] PMID: 39096914 | Gastrointestinal effects of GLP-1 receptor agonists: mechanisms, management, and future directions.
The availability of glucagon-like peptide-1 (GLP-1) receptor agonists (RAs) such as liraglutide and semaglutide, and a GLP-1 and glucose dependent insulinotropic polypeptide coagonist (tirzepatide) represents a paradigm shift in the management of both type 2 diabetes and obesity. There is now considerable attention, including in the public media, on the effect of both long-acting and short-acting GLP-1RAs to delay gastric emptying. Although slowed gastric emptying is integral to reducing post-prandial blood glucose responses in type 2 diabetes, marked sl

In [7]:
# Option A: Hugging Face Inference API (no GPU required)
# Uses the free HF API. Suitable for this hello world.
# Later phases will use the locally fine-tuned model.

from huggingface_hub import InferenceClient

hf_token = os.getenv("HF_TOKEN")
client = InferenceClient(token=hf_token) if hf_token else InferenceClient()

print("Generating response with LLM...\n")

response = client.chat_completion(
    messages=[
        {"role": "user", "content": augmented_prompt}
    ],
    model="meta-llama/Llama-3.1-8B-Instruct",  # Free model for testing
    max_tokens=512,
    temperature=0.1,
    presence_penalty=1.1,
)

print("SYSTEM RESPONSE:\n")
print(response)

Generating response with LLM...

SYSTEM RESPONSE:

ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Based on the provided documents, I can only provide information on the general management of type 2 diabetes, as the specific question about first-line treatments for patients with cardiovascular risk is not directly addressed.\n\nHowever, I can suggest that the management of type 2 diabetes in children and adolescents, as outlined in [2] PMID: 39675348, may provide some insights. According to this document, the management of type 2 diabetes in youth involves a multidisciplinary approach, including lifestyle modifications, pharmacological therapy, and monitoring for comorbidities and complications.\n\nRegarding pharmacological therapy, the document mentions that recently approved therapies for the treatment of youth-onset type 2 diabetes include GLP-1 receptor agonists, such as liraglu

In [10]:
# Option B (alternative): Local model via Unsloth
# Uncomment if you have a GPU and prefer local inference.
# Uses a small model for this test.

from unsloth import FastLanguageModel
import torch

# 1. Configuration
model_id = "microsoft/phi-4-mini-instruct"
max_seq_length = 4096 # You can increase this up to the context limit if doing heavy RAG
dtype = None          # Unsloth automatically detects the best datatype for your RTX 3080
load_in_4bit = True   # The magic line that shrinks the model to ~3.5GB!

print("Loading model and tokenizer via Unsloth...")
# 2. Load Model and Tokenizer natively through Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_id,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Enable Unsloth's native 2x faster inference engine
FastLanguageModel.for_inference(model)

Loading model and tokenizer via Unsloth...
==((====))==  Unsloth 2026.3.11: Fast Phi3 patching. Transformers: 5.3.0.
   \\   /|    NVIDIA GeForce RTX 3080. Num GPUs = 1. Max memory: 9.635 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.6. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/194 [00:00<?, ?it/s]

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating response...


/home/davidsolanas/Documents/MasterIA/TFM/tfm-clinical-ai/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
/home/davidsolanas/Documents/MasterIA/TFM/tfm-clinical-ai/.venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



--- Model Output ---
The provided documents do not contain specific information regarding first-line treatments for type 2 diabetes in patients with cardiovascular risk. To provide an evidence-based response, I would need access to clinical guidelines or research articles that focus on the management of type 2 diabetes in the context of cardiovascular risk. These guidelines typically address the use of medications, lifestyle modifications, and other therapeutic strategies that are considered most effective for this patient population. Since the documents provided do not cover this topic, I am unable to cite a source for the first-line treatments in this specific scenario. For accurate and up-to-date information, please refer to the latest clinical practice guidelines or research articles on the management of type 2 diabetes with cardiovascular risk.


In [23]:
# 4. Your Clinical Prompt
messages = [
    {"role": "user", "content": augmented_prompt}
]

# 5. Apply the model's specific chat format
# We use tokenize=False to get a formatted string back, containing the <user> and <assistant> tags
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# Convert the formatted text into PyTorch tensors and send them to the GPU
inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

print("Generating response...")
# 6. Generate the output tokens
outputs = model.generate(
    **inputs, 
    max_new_tokens=1024,  # Give it plenty of room to explain clinical logic
    temperature=0.0,      # Lower temperature = more focused, factual medical answers
    top_p=0.95            
)

# 7. Decode the generated tokens back into readable text
# We slice the array to remove the prompt from the final output string
response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print("\n--- Model Output ---")
print(response)

Both `max_new_tokens` (=1024) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generating response...

--- Model Output ---
Based on the information provided in the first document, there is a concern that the use of GLP-1 receptor agonists (RAs) such as liraglutide and semaglutide, and the coagonist tirzepatide, may increase the propensity for longer intragastric retention of food. This could potentially increase the risk of aspiration at the time of surgery or upper gastrointestinal endoscopy. The document suggests that this is a significant consideration for the development of pre-procedural guidelines, as the evidence base is currently poor, especially regarding the effect of long-acting GLP-1RAs on gastric emptying. However, the document does not provide a specific risk assessment or statistical data on the incidence of aspiration or other complications related to the use of GLP-1RAs before upper gastrointestinal endoscopy. Therefore, while there is a theoretical risk, the exact risk level is not detailed in the provided document. [PMID: 39096914]


## Checkpoint: What have we validated?

If you reached this point without errors, you have confirmed that:

1. **PubMed API** works -- scientific literature can be downloaded
2. **Embedding model** works -- medical text can be converted to vectors
3. **Qdrant** works -- vectors can be stored and searched
4. **Semantic search** works -- queries retrieve relevant documents
5. **Augmented generation** works -- the LLM generates context-grounded responses

### What we have NOT done yet (and that is expected):
- Fine-tuning -- Phase 2
- Intelligent chunking -- Phase 1
- Formal evaluation -- Phase 4
- Optimised prompts -- Phase 3

### Next steps:
- `01_mtsamples_eda.ipynb` -- Explore the training dataset
- `02_pubmed_ingestion.ipynb` -- Full ingestion pipeline
- `03_embedding_benchmark.ipynb` -- Compare embedding models

---

### Decisions made in this notebook

Log in `DECISIONS.md`:
- Initial embedding model: PubMedBERT (benchmark pending)
- PubMed API via Entrez / BioPython
- Qdrant as the vector store
- RAG prompt format (version 0)

## (Bonus) Log to MLflow

Even for this hello world it is worth logging the experiment.
This builds the habit of tracking from the very beginning.

In [30]:
import mlflow

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("00-vertical-slice")

with mlflow.start_run(run_name="hello_world_v0"):
    # Parameters
    mlflow.log_param("embedding_model", EMBEDDING_MODEL)
    mlflow.log_param("pubmed_query", "type 2 diabetes treatment guidelines 2024")
    mlflow.log_param("num_papers", len(papers))
    mlflow.log_param("top_k", TOP_K)
    mlflow.log_param("vector_dim", VECTOR_DIM)

    # Basic metrics (formal evaluation comes in Phase 4)
    avg_score = sum(h.score for h in results.points) / len(results.points)
    mlflow.log_metric("avg_retrieval_score", avg_score)

    # Save the prompt as an artifact
    with open("/tmp/prompt_v0.txt", "w") as f:
        f.write(augmented_prompt)
    mlflow.log_artifact("/tmp/prompt_v0.txt")

    print("Experiment logged in MLflow")
    print(f"   Avg retrieval score: {avg_score:.4f}")
    print(f"   Dashboard: http://localhost:5000")

2026/03/24 20:56:14 INFO mlflow.tracking.fluent: Experiment with name '00-vertical-slice' does not exist. Creating a new experiment.


Experiment logged in MLflow
   Avg retrieval score: 0.9443
   Dashboard: http://localhost:5000
🏃 View run hello_world_v0 at: http://localhost:5000/#/experiments/1/runs/4dbee055257245798eef0b198bd46e8a
🧪 View experiment at: http://localhost:5000/#/experiments/1
